# Anomalias de Temperatura por Estação

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/local/02_temperature_anomalies.pt.ipynb)

**Objetivo de aprendizagem**: Neste notebook você vai calcular a anomalia de temperatura de cada estação meteorológica em relação à média espacial da rede, tanto na forma tabular/gráfica (`lcz_anomaly`) quanto na forma de uma superfície contínua interpolada por krigagem (`lcz_anomaly_map`), usando Berlim como cidade de referência.

> **Nota de renderização:** todas as figuras deste caderno são geradas por padrão com o renderizador estático nativo do LCZ4py (matplotlib, exportação em 300 dpi PNG/PDF/SVG/TIFF, sem precisar de Plotly/Kaleido). Passe `renderer="plotly"` para uma função de plotagem para obter a figura interativa do Plotly.

## Por que anomalias? Isolando o "efeito urbano" do tempo sinótico

A temperatura do ar bruta em qualquer estação é dominada por padrões meteorológicos de escala sinótica — frentes frias, massas de ar, ciclo sazonal — que afetam todas as estações de uma região quase igualmente. Se compararmos duas estações diretamente pela temperatura absoluta, o sinal climático local (o que realmente nos interessa em climatologia urbana) fica escondido sob esse "ruído" regional que é comum a todas.

A **anomalia** resolve isso: em cada instante de tempo, subtraímos a média espacial de todas as estações da rede do valor observado em cada estação individual. O que sobra é exatamente a componente que não é explicada pelo tempo regional — ou seja, o quanto aquela estação está sistematicamente mais quente ou mais fria que suas vizinhas, controlando para a variabilidade meteorológica do dia. Esse "resíduo espacial" é a assinatura do **efeito urbano local**: cobertura impermeável, ausência de vegetação, fator de visão do céu reduzido (canyons urbanos), rugosidade e capacidade térmica dos materiais construídos.

Esse raciocínio conecta diretamente com o esquema de Local Climate Zones (LCZ) de Stewart & Oke (2012): estações em LCZs compactas e impermeáveis (LCZ 1-3, 8-10) tendem a mostrar anomalias positivas persistentes, enquanto estações em LCZs vegetadas ou com corpos d'água (LCZ 11-17) tendem a mostrar anomalias negativas. O pacote LCZ4r (Anjos et al. 2025, *Scientific Reports*, https://www.nature.com/articles/s41598-025-92000-0) formaliza esse fluxo de trabalho, unindo a classificação LCZ da estação com sua série de anomalias.

Este notebook cobre duas ferramentas complementares:

- **`lcz_anomaly`** — calcula a anomalia média por estação (agregada no tempo) e produz gráficos de ranking (barras divergentes, pontos, "lollipop", dispersão). Ideal para responder "quais estações são consistentemente mais quentes?" e para comunicar resultados de forma direta.
- **`lcz_anomaly_map`** — para cada passo de tempo (ou grupo temporal), krigagem espacial das anomalias pontuais sobre uma grade regular, produzindo um raster contínuo. Usa a classe LCZ como termo de deriva externa (*external drift*) na krigagem quando `LCZinterp=True`, o que ajuda o interpolador a respeitar as fronteiras entre zonas urbanas e rurais mesmo em áreas com poucas estações. Ideal para mapear a extensão espacial completa do efeito, não apenas nos pontos de estação.

## Instalação

Instale o LCZ4py com todos os extras opcionais (krigagem, DuckDB, GeoArrow etc.).

In [ ]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"

## Configuração (repetida do notebook 01)

Como cada notebook desta série deve rodar de forma independente em uma sessão Colab nova, repetimos aqui a preparação: baixar o mapa LCZ de Berlim e carregar os dados de estações meteorológicas (`lcz_data_berlin.csv`, com colunas `date`, `station`, `airT`, `Latitude`, `Longitude`).

In [ ]:
from IPython.display import display
from LCZ4py import lcz_get_map
import pandas as pd

lcz_map_path = lcz_get_map(city="Berlin")
print(lcz_map_path)

df = pd.read_csv("https://raw.githubusercontent.com/ByMaxAnjos/LCZ4py/main/lcz_data_berlin.csv")
df.head()

## `lcz_anomaly` — ranking de estações por anomalia

`lcz_anomaly(x, data_frame, var, station_id, plot_type=..., by=None, iplot=True, ...)` classifica cada estação para uma classe LCZ (a partir de `x`, o raster LCZ), calcula a média temporal do valor de `var` em cada estação e subtrai a média espacial geral de toda a rede — essa diferença é a anomalia. Parâmetros principais:

- `var="airT"`, `station_id="station"` — nomes das colunas de temperatura e identificador de estação.
- `plot_type` — estilo do gráfico: `"diverging_bar"` (padrão, barras horizontais divergentes de zero), `"bar"` (barras verticais coloridas vermelho/azul), `"dot"` (gráfico de pontos de Cleveland conectando média da estação à referência), `"lollipop"` (hastes verticais de zero até a anomalia) ou `"scatter"` (classe LCZ x temperatura média, tamanho do ponto = |anomalia|).
- `by` — opcionalmente facetar por grupo temporal (`"year"`, `"season"`, etc.).
- `iplot=False` — retorna apenas o `pl.DataFrame` com a anomalia, sem plotar.
- `year=`/`month=`/... — filtros de data, iguais aos de `lcz_ts`.

O retorno (quando `iplot=True`) é uma `Figure` do matplotlib — o renderizador estático nativo é o padrão do pacote; use `renderer="plotly"` para a versão interativa do Plotly. Cada estação aparece rotulada com sua classe LCZ e colorida pela paleta oficial. Estações no topo/extremo positivo são "ilhas de calor" locais; no extremo negativo, "ilhas de frescor".

Abaixo demonstramos três estilos de gráfico: `diverging_bar`, `dot` e `scatter`.

In [ ]:
from LCZ4py import lcz_anomaly

fig_div = lcz_anomaly(
    lcz_map_path, df,
    var="airT", station_id="station",
    plot_type="diverging_bar",
    year=2019,
)
display(fig_div)

**Interpretação**: as barras que se estendem para a direita (valores positivos) são estações mais quentes que a média da rede em 2019 — normalmente estações em LCZs compactas/impermeáveis. As que se estendem para a esquerda são mais frias — tipicamente em LCZs vegetadas ou próximas a corpos d'água. A cor de cada barra reflete a classe LCZ da estação.

In [ ]:
fig_dot = lcz_anomaly(
    lcz_map_path, df,
    var="airT", station_id="station",
    plot_type="dot",
    year=2019,
)
display(fig_dot)

**Interpretação**: o gráfico de pontos de Cleveland mostra, para cada estação, sua temperatura média (ponto colorido pela LCZ) conectada por uma linha cinza ao valor de referência (a média espacial geral, ponto cinza). Quanto mais longe o ponto colorido estiver do ponto de referência, maior a anomalia daquela estação — essa visualização é útil para comparar magnitudes absolutas de temperatura, não só o desvio.

In [ ]:
fig_scatter = lcz_anomaly(
    lcz_map_path, df,
    var="airT", station_id="station",
    plot_type="scatter",
    year=2019,
)
display(fig_scatter)

**Interpretação**: aqui o eixo X é a classe LCZ e o eixo Y é a temperatura média da estação; o tamanho do marcador escala com |anomalia|. Isso revela rapidamente se a anomalia está associada de forma consistente à classe LCZ (esperado, segundo a teoria de Stewart & Oke) ou se há estações "atípicas" dentro da sua própria classe — possivelmente por má exposição do sensor, microtopografia local, ou proximidade a fontes de calor antropogênico não capturadas na classificação LCZ.

## `lcz_anomaly_map` — superfície espacial contínua de anomalia

Enquanto `lcz_anomaly` responde "qual estação é mais quente", `lcz_anomaly_map` responde "qual é o padrão espacial completo da anomalia na cidade inteira" — produzindo um raster contínuo via krigagem paralela, em vez de valores discretos nos pontos de estação.

Para cada passo de tempo (ou grupo, se `by` for definido), a função:
1. calcula a anomalia pontual de cada estação (valor − média espacial da rede naquele instante);
2. interpola essas anomalias pontuais sobre uma grade regular via krigagem (`method="krige"`, usando o modelo de variograma `vg_model`), com a classe LCZ como covariável de deriva externa quando `LCZinterp=True` e há pelo menos 2 classes LCZ distintas entre as estações;
3. salva um GeoTIFF multibanda (uma banda por passo de tempo).

Como a krigagem paralela sobre muitas estações/passos de tempo é pesada, usamos aqui uma resolução espacial grosseira (`sp_res=500` m) e uma janela temporal curta (`month=1, year=2019`, ~744 passos horários) para manter a execução rápida neste tutorial. Em um estudo real, `sp_res` menor (50-100 m) e um período mais longo dariam mais detalhe, ao custo de tempo de execução.

In [6]:
from LCZ4py import lcz_anomaly_map

anomaly_map_path = lcz_anomaly_map(
    lcz_map_path, df,
    var="airT", station_id="station",
    sp_res=500.0,
    tp_res="hour",
    method="krige",
    LCZinterp=True,
    year=2019, month=1,
    n_jobs=-1,
)
print(anomaly_map_path)

/var/folders/x0/lh_r13_n6_gd2h16vhk2s_3r0000gn/T/tmpf2rodqyf.tif


### Visualizando o raster de anomalia

O GeoTIFF resultante tem uma banda por passo de tempo. Usamos `lcz_plot_map(..., data_type="continuous")` para exibir uma delas como um mapa de calor contínuo (em vez do modo categórico usado para mapas de classe LCZ).

In [ ]:
from LCZ4py import lcz_plot_map

result = lcz_plot_map(
    anomaly_map_path,
    data_type="continuous",
    band=1,
    colorscale="RdBu_r",
    title="Anomalia espacial de temperatura do ar (Berlim, banda 1)",
)
display(result.fig)

**Interpretação**: tons quentes (vermelho) indicam áreas onde a interpolação prevê temperatura acima da média espacial da rede naquele instante — geralmente sobre o núcleo urbano compacto; tons frios (azul) aparecem sobre parques, florestas e corpos d'água. Como a krigagem usa a classe LCZ como deriva externa, as fronteiras entre zonas tendem a aparecer mais nítidas do que em uma interpolação puramente geográfica (IDW/RBF), refletindo a estrutura urbana subjacente em vez de apenas a distância às estações.

**Tabular vs. espacial — quando usar qual**: `lcz_anomaly` é a ferramenta certa quando você quer um **ranking objetivo de estações** (por exemplo, para relatórios ou para identificar quais sensores merecem inspeção de exposição). `lcz_anomaly_map` é a ferramenta certa quando você precisa de uma **superfície contínua** — por exemplo, para estimar a anomalia em um local sem estação, para sobrepor com uso do solo, ou para produzir um mapa de comunicação pública do efeito de ilha de calor.

## Conclusão

Neste notebook você aprendeu a isolar o efeito urbano local do tempo regional através da técnica de anomalia: subtraindo a média espacial da rede em cada instante, o que resta é o sinal persistente e específico de cada local — a assinatura da forma urbana (LCZ) sobre a temperatura. Vimos a versão tabular/gráfica (`lcz_anomaly`, útil para ranquear estações) e a versão espacialmente interpolada (`lcz_anomaly_map`, útil para mapear a cidade inteira via krigagem com deriva externa LCZ).

**Notebook anterior**: `01_lcz_time_series` — séries temporais estratificadas por LCZ.
**Próximo notebook**: `03_urban_heat_island` — quantificação direta da intensidade da Ilha de Calor Urbana (UHI) comparando estações urbanas e rurais.

## Figuras de anomalia prontas para publicação com `style`

As funções de plotagem agora aceitam `style="default"`, `style="nature"`, `style="science"` e `style="generic_bw"`. Os presets de revista ajustam a família de fontes, o tamanho físico em milímetros, o DPI (300 para os estilos voltados à publicação) e o tratamento de cores, para que tanto os gráficos de ranking de estações quanto os mapas contínuos de anomalia possam ser exportados de forma limpa para publicação.

Para o painel de mapa, também estão disponíveis os auxiliares cartográficos automáticos: `add_scalebar=True` e `add_north_arrow=True`.

Um exemplo curto pronto para publicação:

- `style="nature"` para o mapa de anomalia.
- `isave=True` com `save_extension="png"` ou `"pdf"` para gravar a figura final.


In [ ]:
from LCZ4py import lcz_anomaly, lcz_plot_map

fig_anom_pub = lcz_anomaly(
    x=lcz_map_path,
    data_frame=df,
    var="airT",
    station_id="station",
    plot_type="dot",
    style="science",
    isave=True,
    save_extension="png",
    lang="pt",
)
display(fig_anom_pub)

anom_map_pub = lcz_plot_map(
    anomaly_map_path,
    data_type="continuous",
    band=1,
    title="Anomalia de temperatura — estilo de publicação",
    style="nature",
    isave=True,
    save_extension="pdf",
    add_scalebar=True,
    add_north_arrow=True,
)
display(anom_map_pub.fig)
